# Semantic article segmentation

Segment an English or Vietnamese Markdown article, inspect its sections, then generate headings independently.

Run `uv sync --extra offline` and select this project's `.venv` kernel. Restart the kernel after package changes. The notebook loads the project `.env`. Automatic selection uses OpenAI embeddings when `OPENAI_API_KEY` is nonblank; otherwise it uses local multilingual E5. Local weights may download on first use. Local segmentation needs no API key; OpenAI heading generation does.

In [1]:
import os
from pathlib import Path
from time import perf_counter

from dotenv import load_dotenv
from IPython.display import Markdown, display

from data_platform.digest.segmenter import (
    detect_document_structure,
    generate_headings,
    render_markdown,
    resolve_embedding_model,
    segment_article,
    segment_article_semantic,
)

project_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src/data_platform").is_dir()
)
load_dotenv(project_root / ".env", override=False)

True

## Choose the article and models

Leave `embedding_model=None` for automatic selection. Explicit overrides include `openai:text-embedding-3-small` and `sentence-transformers:intfloat/multilingual-e5-small`. Set `local_files_only=True` to forbid model downloads, and optionally set `local_model_path` to a prepared E5 model directory. These local settings apply when a local backend is selected. Provider errors are reported without switching models.

In [10]:
input_path = project_root / ".data" / "article-vi-01.md"
embedding_model = None
local_files_only = False
local_model_path = None
decision_model = "openai:gpt-6-luna"
heading_model = "openai:gpt-6-luna"
max_concurrency = 4
output_path = project_root / ".data" / "segmented" / f"{input_path.stem}.md"

markdown_doc = input_path.read_text(encoding="utf-8")
selected_backend = resolve_embedding_model(embedding_model)
print(f"Article: {input_path.name}; embedding backend: {selected_backend}")

Article: article-vi-01.md; embedding backend: openai:text-embedding-3-small


## 1. Detect document structure

Existing Markdown headings and publication metadata are recognized locally. Rules select standalone candidates with at most one sentence, including short labels such as ABOUT VINGROUP; a typed model decision confirms whether each candidate is a heading. Classification uses the decision model when credentials are available and reports when it is skipped. Existing heading wording is preserved.

In [11]:
structure = await detect_document_structure(
    markdown_doc, decision_model=decision_model, max_concurrency=max_concurrency
)
print("Classification skipped:", structure.classification_skipped)
for candidate in structure.candidates:
    print(candidate.is_heading, repr(candidate.text.strip()))
print("Structural blocks:", [(b.kind, b.start, b.end) for b in structure.blocks])

Classification skipped: False
Structural blocks: [('title', 0, 99), ('metadata', 100, 125)]


## 2. Segment the article

This cell makes embedding requests or runs the local model. It does not generate headings. Source text, quotes, and paragraph-aligned offsets are preserved.

The title, metadata, and confirmed headings are kept out of embedding comparisons. Sections never cross a heading boundary.

Each attributed utterance gets a separate quote section, including repeated quotations from the same speaker. Names and reporting clauses stay with the quote. Interrupted speech stays together; ordinary quoted terms stay in prose. The English fixture contains three attributed quotations.

In [12]:
started = perf_counter()
sections = await segment_article_semantic(
    markdown_doc,
    structure=structure,
    embedding_model=embedding_model,
    local_files_only=local_files_only,
    local_model_path=local_model_path,
)
assert "".join(section.text for section in sections) == markdown_doc
print(f"{len(sections)} sections in {perf_counter() - started:.2f}s")
for index, section in enumerate(sections):
    print(f"[{index}] {section.kind}: characters {section.start}–{section.end}")
    print(section.text[:180], "\n")
print("Independent quotations:", sum(s.kind == "quote" for s in sections))

6 sections in 2.24s
[0] title: characters 0–100
# CỤC ĐĂNG KIỂM VIỆT NAM PHỐI HỢP VỚI VINFAST TỔ CHỨC KHÓA ĐÀO TẠO NÂNG CAO NĂNG LỰC KIỂM THỬ Ô TÔ

 

[1] metadata: characters 100–126
Ngày đăng bài 22-09-2026

 

[2] prose: characters 126–2663
Từ ngày 7/9 - 11/9/2026, tại Trung tâm Thử nghiệm VinFast (Hải Phòng), Cục Đăng kiểm Việt Nam phối hợp với VinFast và Trung tâm Quốc tế hóa Tiêu chuẩn Ô tô Nhật Bản (JASIC) tổ chức 

[3] prose: characters 2663–3067
Hệ thống thử nghiệm Xe trượt gia tốc ngược (Reverse Acceleration Sled Test - RAST) cho phép thực hiện các bài thử và nghiên cứu chuyên sâu cho ghế ngồi, dây đai an toàn và hệ thống 

[4] prose: characters 3067–4440
Một phòng hiệu chuẩn hình nộm với trang thiết bị hiện đại hàng đầu thế giới cũng được đầu tư để làm chủ quá trình vận hành. Bên cạnh đó, Trung tâm còn được trang bị hệ thống đầy đủ 

[5] prose: characters 4440–4920
Việc đầu tư cơ sở vật chất bài bản, đồng bộ cho hệ thống kiểm thử, đồng thời tham gia sâu vào các hoạt động đào

## Optional: compare vectorization methods

Set `run_comparisons=True` to compare counts, TF-IDF, and explicit local/OpenAI embedding models. This may download local weights and make additional paid embedding requests. The comparison does not replace `sections` selected above. Inspect boundaries instead of judging quality by section count alone.

In [13]:
run_comparisons = False
if run_comparisons:
    comparisons = {}
    for vectorizer in ("counts", "tfidf"):
        started = perf_counter()
        result = segment_article(
            markdown_doc, vectorizer=vectorizer, structure=structure
        )
        comparisons[vectorizer] = result
        print(vectorizer, f"{perf_counter() - started:.2f}s", [s.start for s in result])
    comparison_models = ["sentence-transformers:intfloat/multilingual-e5-small"]
    if os.getenv("OPENAI_API_KEY", "").strip():
        comparison_models.append("openai:text-embedding-3-small")
    for name in comparison_models:
        started = perf_counter()
        result = await segment_article_semantic(
            markdown_doc,
            embedding_model=name,
            structure=structure,
            local_files_only=local_files_only,
            local_model_path=local_model_path,
        )
        comparisons[name] = result
        print(name, f"{perf_counter() - started:.2f}s", [s.start for s in result])

## 3. Generate headings

Headings use complete sentences in the article's language, summarize the first substantive claim, preserve action status, and use named entities with supplied honorifics and full titles. Quote headings use neutral indirect speech and explicit attribution. Full article context is supplied for reference resolution.

This cell makes up to four heading requests concurrently and preserves section order. Without an OpenAI key, it leaves sections unheaded so local segmentation and saving still work. Set credentials and rerun setup and this cell to generate headings.

The title and publication metadata appear before any generated subtitle. Existing headings keep their wording, and the first content section below each existing heading does not receive a duplicate generated heading.

In [14]:
if heading_model.startswith("openai:") and not os.getenv("OPENAI_API_KEY", "").strip():
    headed_sections = sections
    print("Headings skipped: set OPENAI_API_KEY to generate OpenAI headings.")
else:
    headed_sections = await generate_headings(
        sections,
        article_text=markdown_doc,
        model=heading_model,
        max_concurrency=max_concurrency,
    )
    for index, section in enumerate(headed_sections):
        print(f"[{index}] {section.heading}")

[0] None
[1] None
[2] Cục Đăng kiểm Việt Nam, VinFast và JASIC phối hợp tổ chức khóa đào tạo quy định kỹ thuật của Liên Hợp Quốc.
[3] Hệ thống RAST cho phép thực hiện các bài thử và nghiên cứu chuyên sâu về ghế ngồi, dây đai an toàn và hệ thống túi khí.
[4] Trung tâm Thử nghiệm VinFast đã đầu tư phòng hiệu chuẩn hình nộm với trang thiết bị hiện đại hàng đầu thế giới.
[5] Việc đầu tư cơ sở vật chất cho hệ thống kiểm thử và tham gia đào tạo, trao đổi kỹ thuật quốc tế cho thấy VinFast theo đuổi định hướng phát triển sản phẩm đề cao chất lượng và an toàn.


## Render Markdown

Rendering is local. Sections without generated headings retain their original text.

In [15]:
segmented_doc = render_markdown(headed_sections)
display(Markdown(segmented_doc))

# CỤC ĐĂNG KIỂM VIỆT NAM PHỐI HỢP VỚI VINFAST TỔ CHỨC KHÓA ĐÀO TẠO NÂNG CAO NĂNG LỰC KIỂM THỬ Ô TÔ

Ngày đăng bài 22-09-2026

## Cục Đăng kiểm Việt Nam\, VinFast và JASIC phối hợp tổ chức khóa đào tạo quy định kỹ thuật của Liên Hợp Quốc\.

Từ ngày 7/9 - 11/9/2026, tại Trung tâm Thử nghiệm VinFast (Hải Phòng), Cục Đăng kiểm Việt Nam phối hợp với VinFast và Trung tâm Quốc tế hóa Tiêu chuẩn Ô tô Nhật Bản (JASIC) tổ chức khóa đào tạo các quy định kỹ thuật của Liên Hợp Quốc (UN Regulations) dành cho cán bộ quản lý của các đơn vị kiểm thử, chứng nhận an toàn xe cơ giới. Sự kiện không chỉ cho thấy vai trò và đóng góp ngày càng lớn của Việt Nam trong việc nâng cao chất lượng, tiêu chuẩn an toàn kỹ thuật trong ngành công nghiệp sản xuất ô tô, mà còn cho thấy định hướng phát triển sản phẩm đề cao chất lượng và an toàn của VinFast, cũng như khẳng định năng lực kiểm thử đẳng cấp quốc tế của Trung tâm Thử nghiệm VinFast.

Tham gia khóa đào tạo gồm các chuyên gia, cán bộ quản lý tại các đơn vị kỹ thuật, trung tâm thử nghiệm an toàn ô tô tại Việt Nam và các quốc gia trong khu vực châu Á. Chương trình được chủ trì thực hiện bởi Cục Đăng kiểm Việt Nam; VinFast cung cấp cơ sở vật chất, các điều kiện kỹ thuật cần thiết cho toàn bộ chương trình và trực tiếp đào tạo lý thuyết, thực hành thử nghiệm an toàn va chạm theo quy chuẩn UN R94 (va chạm trước) và UN R95 (va chạm bên hông); Trung tâm Quốc tế hóa Tiêu chuẩn Ô tô Nhật Bản (JASIC) đào tạo về hai quy chuẩn UN R100 (an toàn hệ thống điện áp cao) và UN R154 (đánh giá tiêu thụ năng lượng theo chu trình WLTP), đồng thời cấp chứng nhận hoàn thành khóa đào tạo cho các thành viên tham gia.

Khóa đào tạo có ý nghĩa đặc biệt quan trọng đối với các tổ chức, cá nhân đang hoạt động trong lĩnh vực kiểm thử, chứng nhận an toàn và phê duyệt kiểu loại xe cơ giới của Việt Nam và các nước trong khu vực châu Á. Chương trình giúp nâng cao năng lực cán bộ, trang bị các kiến thức, kỹ năng cần thiết để sẵn sàng áp dụng các quy định kỹ thuật của Liên Hợp Quốc và quy trình thử nghiệm tiên tiến, trong bối cảnh ngành công nghiệp ô tô Việt Nam và khu vực đang chuyển dịch mạnh mẽ sang các loại hình phương tiện xanh sử dụng điện.

Đóng góp quan trọng vào thành công của chương trình là sự hỗ trợ về cơ sở vật chất, trang thiết bị hiện đại và đội ngũ kỹ sư chuyên nghiệp của VinFast. Đây là cơ sở được đầu tư đồng bộ hệ thống phòng thí nghiệm, trang thiết bị và các điều kiện kỹ thuật đáp ứng yêu cầu thử nghiệm theo tiêu chuẩn quốc tế. Trong đó, nổi bật là Trung tâm Kiểm thử An toàn xe với hệ thống đường ray thử nghiệm va chạm dài 250 m, hệ thống điều khiển hiện đại, cho phép thực hiện nhiều bài thử như va chạm trực diện, va chạm lệch tâm, va chạm xe với xe, va chạm bên, va chạm cột, va chạm phía sau… với tốc độ lên tới 120 km/h.

## Hệ thống RAST cho phép thực hiện các bài thử và nghiên cứu chuyên sâu về ghế ngồi\, dây đai an toàn và hệ thống túi khí\.

Hệ thống thử nghiệm Xe trượt gia tốc ngược (Reverse Acceleration Sled Test - RAST) cho phép thực hiện các bài thử và nghiên cứu chuyên sâu cho ghế ngồi, dây đai an toàn và hệ thống túi khí. Trung tâm cũng được trang bị toàn bộ hình nộm thử nghiệm, đầy đủ chủng loại cho yêu cầu thử nghiệm toàn cầu, từ hình nộm dành cho người lớn đến trẻ em, tiêu biểu như Hybrid III, THOR, EuroSID, WorldSID, Q-Series.

## Trung tâm Thử nghiệm VinFast đã đầu tư phòng hiệu chuẩn hình nộm với trang thiết bị hiện đại hàng đầu thế giới\.

Một phòng hiệu chuẩn hình nộm với trang thiết bị hiện đại hàng đầu thế giới cũng được đầu tư để làm chủ quá trình vận hành. Bên cạnh đó, Trung tâm còn được trang bị hệ thống đầy đủ các rào cản cho yêu cầu thử nghiệm toàn cầu, tiêu biểu như ODB, MPDB, SOB, MDB, AE-MDB, IIHS MDB; khoảng 1.000 cảm biến gia tốc cùng hơn 500 kênh thu thập dữ liệu; hệ thống đèn chiếu sáng tốc độ cao; hệ thống camera tốc độ cao giúp ghi nhận đầy đủ diễn biến của thử nghiệm đã và đang được vận hành.

Với hệ thống thử nghiệm va chạm toàn xe trong nhà và ngoài trời, hệ thống xe trượt gia tốc ngược, phòng hiệu chuẩn hình nộm thử nghiệm va chạm cùng hệ thống thiết bị đo lường hiện đại, Trung tâm có thể triển khai đầy đủ các bài thử theo yêu cầu của các chương trình đánh giá an toàn trên toàn cầu như UN ECE, FMVSS, AIS… và các yêu cầu đánh giá khắt khe hơn để xếp hạng sao trên thế giới như Euro NCAP, US NCAP, ASEAN NCAP, Bharat NCAP…

Không chỉ phục vụ nhu cầu thử nghiệm của VinFast nhằm đảm bảo chất lượng, tiêu chuẩn an toàn cao nhất, Trung tâm Thử nghiệm VinFast còn hướng đến mục tiêu trở thành trung tâm thử nghiệm hàng đầu của khu vực Đông Nam Á và là địa chỉ tin cậy cho các hoạt động thử nghiệm, đánh giá chất lượng phương tiện của nhiều đối tác trong và ngoài nước, đồng thời mở ra những cơ hội hợp tác quốc tế sâu rộng hơn trong lĩnh vực thử nghiệm an toàn và chứng nhận ô tô.

## Việc đầu tư cơ sở vật chất cho hệ thống kiểm thử và tham gia đào tạo\, trao đổi kỹ thuật quốc tế cho thấy VinFast theo đuổi định hướng phát triển sản phẩm đề cao chất lượng và an toàn\.

Việc đầu tư cơ sở vật chất bài bản, đồng bộ cho hệ thống kiểm thử, đồng thời tham gia sâu vào các hoạt động đào tạo, trao đổi kinh nghiệm kỹ thuật quốc tế cũng cho thấy định hướng phát triển sản phẩm đề cao chất lượng và an toàn của VinFast. Không chỉ đẩy mạnh nghiên cứu - phát triển để mang đến cho người tiêu dùng những sản phẩm ngày càng tốt hơn, VinFast còn đang góp phần xây dựng nền móng vững chắc cho sự phát triển lâu dài của ngành công nghiệp ô tô Việt Nam và khu vực./.

## Save

This replaces the generated output file while preserving the input article.

In [16]:
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(segmented_doc, encoding="utf-8")
print(f"Saved: {output_path}")

Saved: /Users/chuminhquan/projects/data-platform/.data/segmented/article-vi-01.md
